In [1]:
# 00_cache_data.ipynb
# Run this ONCE. Caches all data to /tmp/ for all subsequent notebooks.

import numpy as np
import os
import re
import glob
import pickle
from tqdm import tqdm

# ── Source paths (Orion / shapeSimilarity; override if needed) ───────────────
ENC_10K = "/mnt/data1/shapeSimilarity/encodings/pk-real10k0.002"
GT_10K_DIR = "/mnt/data1/shapeSimilarity/warehouse/pk-query-10k"

# Legacy DGX paths (used if shapeSimilarity dirs are missing)
ENC_10K_RAID = "/raid/ruban/encodings/pk-real10k0.002"
GT_10K_RAID = "/raid/ruban/groundtruth/pk-query-10k"
GT_FULL_RAID = "/raid/ruban/groundtruth/pk-query-187019"

enc_10k_dir = ENC_10K if os.path.isdir(ENC_10K) else ENC_10K_RAID
gt_10k_dir = GT_10K_DIR if os.path.isdir(GT_10K_DIR) else GT_10K_RAID
print(f"10k encodings: {enc_10k_dir}")
print(f"10k GT:       {gt_10k_dir}")


def list_encoding_files(enc_dir):
    """Sort real_<id>.txt numerically so row index matches polygon id."""
    paths = glob.glob(os.path.join(enc_dir, "real_*.txt"))
    def sort_key(p):
        m = re.search(r"real_(\d+)\.txt", os.path.basename(p))
        return int(m.group(1)) if m else 0
    return sorted(paths, key=sort_key)

# ─── 1. Full quadtree (already cached) ───────────────────────────────────────
print("=== Full quadtree ===")
if os.path.exists('/tmp/qtree_vectors_full.npy'):
    qt_full = np.load('/tmp/qtree_vectors_full.npy')
    print(f"Already cached: {qt_full.shape}")
else:
    print("Not found — run original pipeline first")

# ─── 2. 10k quadtree ─────────────────────────────────────────────────────────
print("\n=== 10k quadtree ===")
if os.path.exists('/tmp/qt_10k.npy'):
    qt_10k = np.load('/tmp/qt_10k.npy')
    print(f"Already cached: {qt_10k.shape}")
else:
    enc_files = list_encoding_files(enc_10k_dir)
    print(f"Loading {len(enc_files)} files from {enc_10k_dir} ...")
    arrays = []
    for fpath in tqdm(enc_files):
        arr = np.loadtxt(fpath, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        arrays.append(arr)
    qt_10k = np.vstack(arrays)
    np.save('/tmp/qt_10k.npy', qt_10k)
    print(f"Cached: {qt_10k.shape}  (expect ~10000 x 18499)")

# ─── 3. Full GT ───────────────────────────────────────────────────────────────
print("\n=== Full GT ===")
if os.path.exists('/tmp/gt_lookup_full.pkl'):
    with open('/tmp/gt_lookup_full.pkl', 'rb') as f:
        gt_full = pickle.load(f)
    print(f"Already cached: {len(gt_full)} queries")
else:
    gt_full = {}
    gt_dir  = GT_FULL_RAID
    for fname in tqdm(sorted(os.listdir(gt_dir)), desc="Full GT"):
        fpath = os.path.join(gt_dir, fname)
        with open(fpath, 'r') as f:
            content = f.read().strip()
        for line in content.split('\n'):
            line = line.strip()
            if not line: continue
            ids = [int(x.strip()) for x in line.split(',')
                   if x.strip().lstrip('-').isdigit()]
            if len(ids) >= 2:
                gt_full[ids[0]] = ids[1:]
    with open('/tmp/gt_lookup_full.pkl', 'wb') as f:
        pickle.dump(gt_full, f)
    print(f"Cached: {len(gt_full)} queries")

# ─── 4. 10k GT ────────────────────────────────────────────────────────────────
print("\n=== 10k GT ===")
if os.path.exists('/tmp/gt_lookup_10k.pkl'):
    with open('/tmp/gt_lookup_10k.pkl', 'rb') as f:
        gt_10k = pickle.load(f)
    print(f"Already cached: {len(gt_10k)} queries")
else:
    gt_10k = {}
    gt_dir = gt_10k_dir
    for fname in tqdm(sorted(os.listdir(gt_dir)), desc="10k GT"):
        fpath = os.path.join(gt_dir, fname)
        with open(fpath, 'r') as f:
            content = f.read().strip()
        for line in content.split('\n'):
            line = line.strip()
            if not line: continue
            ids = [int(x.strip()) for x in line.split(',')
                   if x.strip().lstrip('-').isdigit()]
            if len(ids) >= 2:
                gt_10k[ids[0]] = ids[1:]
    with open('/tmp/gt_lookup_10k.pkl', 'wb') as f:
        pickle.dump(gt_10k, f)
    print(f"Cached: {len(gt_10k)} queries")

# ─── Summary ──────────────────────────────────────────────────────────────────
print("\n=== Cache summary ===")
cache_files = [
    '/tmp/qtree_vectors_full.npy',
    '/tmp/qt_10k.npy',
    '/tmp/gt_lookup_full.pkl',
    '/tmp/gt_lookup_10k.pkl',
]
for f in cache_files:
    if os.path.exists(f):
        print(f"  OK  {os.path.getsize(f)/1024**2:8.1f} MB — {f}")
    else:
        print(f"  MISSING — {f}")

=== Full quadtree ===
Already cached: (233773, 18220)

=== 10k quadtree ===
Already cached: (10000, 18499)

=== Full GT ===
Already cached: 44666 queries

=== 10k GT ===
Already cached: 1818 queries

=== Cache summary ===
  OK   16248.1 MB — /tmp/qtree_vectors_full.npy
  OK     705.7 MB — /tmp/qt_10k.npy
  OK     887.1 MB — /tmp/gt_lookup_full.pkl
  OK       1.1 MB — /tmp/gt_lookup_10k.pkl
